# 04 - Creación de tablas Hive - Catastro DNC

En este notebook se crean las tablas externas Hive sobre el modelo analítico guardado en la zona curated del datalake.

El modelo fue construido previamente en `/cur/obligatorio_catastro` y contiene dimensiones y tablas de hechos en formato Parquet.

El objetivo de esta etapa es exponer el modelo a través de Hive para que las preguntas analíticas posteriores se respondan consultando tablas Hive, tal como solicita la consigna del obligatorio.

## 1. Inicialización del entorno Spark con soporte Hive

Se inicializa una sesión Spark con soporte para Hive, necesaria para crear bases, tablas externas y ejecutar consultas SQL sobre el modelo analítico.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Obligatorio Catastro - Hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-05T03:58:19,109 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Definición de rutas y base Hive

Se define la ruta del modelo curated en HDFS y el nombre de la base de datos Hive que contendrá las tablas externas del modelo analítico.

In [2]:
CUR_PATH = "/cur/obligatorio_catastro"
HIVE_DB = "obligatorio_catastro"

print("CUR_PATH:", CUR_PATH)
print("HIVE_DB:", HIVE_DB)

CUR_PATH: /cur/obligatorio_catastro
HIVE_DB: obligatorio_catastro


## 3. Creación de base de datos Hive

Se crea una base de datos Hive específica para el obligatorio.  
Todas las tablas externas del modelo analítico serán creadas dentro de esta base.

In [3]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {HIVE_DB}")
spark.sql(f"USE {HIVE_DB}")

spark.sql("SHOW DATABASES").show(truncate=False)

2026-06-05T03:58:26,252 INFO [Thread-4] org.apache.hadoop.hive.conf.HiveConf - Found configuration file file:/home/ort/spark/conf/hive-site.xml
2026-06-05T03:58:26,479 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.metastore.wm.default.pool.size does not exist
2026-06-05T03:58:26,479 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.llap.task.scheduler.preempt.independent does not exist
2026-06-05T03:58:26,479 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.llap.output.format.arrow does not exist
2026-06-05T03:58:26,479 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.tez.llap.min.reducer.per.executor does not exist
2026-06-05T03:58:26,479 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.arrow.root.allocator.limit does not exist
2026-06-05T03:58:26,479 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.vectorized.use.che

## 4. Creación de tablas externas Hive

En esta sección se crean tablas externas Hive apuntando a los archivos Parquet almacenados en la zona curated.

Las tablas son externas porque los datos permanecen físicamente en HDFS bajo `/cur/obligatorio_catastro`, mientras que Hive almacena la definición de esquema y ubicación.

In [4]:
def crear_tabla_externa_parquet(nombre_tabla, path):
    """
    Crea una tabla externa Hive a partir de archivos Parquet existentes en HDFS.
    Primero lee el Parquet con Spark para obtener el esquema, luego crea la tabla
    usando ese esquema y la ubicación indicada.
    """
    df = spark.read.parquet(path)
    
    # Borrar tabla si existe para poder recrearla limpiamente
    spark.sql(f"DROP TABLE IF EXISTS {HIVE_DB}.{nombre_tabla}")
    
    # Crear tabla externa usando el esquema del dataframe y la ubicación HDFS
    (
        df.write
        .mode("overwrite")
        .option("path", path)
        .format("parquet")
        .saveAsTable(f"{HIVE_DB}.{nombre_tabla}")
    )
    
    print(f"Tabla creada: {HIVE_DB}.{nombre_tabla} -> {path}")

## 5. Definición de tablas del modelo curated

Se define la lista de tablas del modelo analítico construido en la zona curated.

Cada tabla Hive apuntará a una carpeta Parquet existente dentro de `/cur/obligatorio_catastro`.

In [5]:
tablas_curated = [
    "dim_departamento",
    "dim_localidad",
    "dim_destino",
    "dim_categoria_construccion",
    "dim_estado_conservacion",
    "dim_cubierta",
    "dim_cielorraso",
    "dim_tipo_obra",
    "fact_padrones_urbanos",
    "fact_padrones_rurales",
    "fact_lineas_construccion",
    "fact_historico_valores"
]

for tabla in tablas_curated:
    print(f"{tabla} -> {CUR_PATH}/{tabla}")

dim_departamento -> /cur/obligatorio_catastro/dim_departamento
dim_localidad -> /cur/obligatorio_catastro/dim_localidad
dim_destino -> /cur/obligatorio_catastro/dim_destino
dim_categoria_construccion -> /cur/obligatorio_catastro/dim_categoria_construccion
dim_estado_conservacion -> /cur/obligatorio_catastro/dim_estado_conservacion
dim_cubierta -> /cur/obligatorio_catastro/dim_cubierta
dim_cielorraso -> /cur/obligatorio_catastro/dim_cielorraso
dim_tipo_obra -> /cur/obligatorio_catastro/dim_tipo_obra
fact_padrones_urbanos -> /cur/obligatorio_catastro/fact_padrones_urbanos
fact_padrones_rurales -> /cur/obligatorio_catastro/fact_padrones_rurales
fact_lineas_construccion -> /cur/obligatorio_catastro/fact_lineas_construccion
fact_historico_valores -> /cur/obligatorio_catastro/fact_historico_valores


## 6. Creación de tablas externas Hive

Se crean tablas Hive sobre los archivos Parquet del modelo curated.

Para cada tabla:

1. Se elimina la tabla Hive si ya existía.
2. Se crea nuevamente apuntando a la ubicación correspondiente en HDFS.
3. Se utiliza formato Parquet.

In [6]:
for tabla in tablas_curated:
    path = f"{CUR_PATH}/{tabla}"
    
    print(f"Creando tabla Hive: {HIVE_DB}.{tabla}")
    print(f"Ubicación: {path}")
    
    spark.sql(f"DROP TABLE IF EXISTS {HIVE_DB}.{tabla}")
    
    spark.sql(f"""
        CREATE TABLE {HIVE_DB}.{tabla}
        USING PARQUET
        LOCATION '{path}'
    """)
    
    print(f"OK: {HIVE_DB}.{tabla}")

Creando tabla Hive: obligatorio_catastro.dim_departamento
Ubicación: /cur/obligatorio_catastro/dim_departamento


2026-06-05T03:58:31,183 INFO [Thread-4] org.apache.hadoop.hive.ql.security.authorization.plugin.sqlstd.SQLStdHiveAccessController - Created SQLStdHiveAccessController for session context : HiveAuthzSessionContext [sessionString=463d9ac6-e41b-4cc4-a231-cad59ee48cdd, clientType=HIVECLI]
2026-06-05T03:58:31,185 WARN [Thread-4] org.apache.hadoop.hive.ql.session.SessionState - METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-06-05T03:58:31,186 INFO [Thread-4] hive.metastore - Mestastore configuration hive.metastore.filter.hook changed from org.apache.hadoop.hive.metastore.DefaultMetaStoreFilterHookImpl to org.apache.hadoop.hive.ql.security.authorization.plugin.AuthorizationMetaStoreFilterHook
2026-06-05T03:58:31,190 INFO [Thread-4] hive.metastore - Closed a connection to metastore, current connections: 0
2026-06-05T03:58:31,192 INFO [Thread-4] hive.metastore - Trying to connect to metastore with URI thrift://l

## 7. Validación de tablas Hive creadas

Se valida que las tablas externas Hive fueron creadas correctamente dentro de la base `obligatorio_catastro`.

Para ello se revisa:

- Listado de tablas disponibles.
- Esquema de algunas tablas.
- Cantidad de registros por tabla.

### 7.1 — Mostramos tablas creadas

In [7]:
spark.sql(f"USE {HIVE_DB}")

spark.sql("SHOW TABLES").show(50, truncate=False)

+--------------------+--------------------------+-----------+
|namespace           |tableName                 |isTemporary|
+--------------------+--------------------------+-----------+
|obligatorio_catastro|dim_categoria_construccion|false      |
|obligatorio_catastro|dim_cielorraso            |false      |
|obligatorio_catastro|dim_cubierta              |false      |
|obligatorio_catastro|dim_departamento          |false      |
|obligatorio_catastro|dim_destino               |false      |
|obligatorio_catastro|dim_estado_conservacion   |false      |
|obligatorio_catastro|dim_localidad             |false      |
|obligatorio_catastro|dim_tipo_obra             |false      |
|obligatorio_catastro|fact_historico_valores    |false      |
|obligatorio_catastro|fact_lineas_construccion  |false      |
|obligatorio_catastro|fact_padrones_rurales     |false      |
|obligatorio_catastro|fact_padrones_urbanos     |false      |
+--------------------+--------------------------+-----------+



### 7.2 — Describir algunas tablas

In [8]:
from pyspark.sql import Row

validacion_hive = []

for tabla in tablas_curated:
    cantidad = spark.sql(f"SELECT COUNT(*) AS cantidad FROM {HIVE_DB}.{tabla}").collect()[0]["cantidad"]
    
    validacion_hive.append(
        Row(
            tabla=tabla,
            cantidad_registros=cantidad
        )
    )

df_validacion_hive = spark.createDataFrame(validacion_hive)

df_validacion_hive.orderBy("tabla").show(truncate=False)

[Stage 50:>                                                         (0 + 2) / 2]

+--------------------------+------------------+
|tabla                     |cantidad_registros|
+--------------------------+------------------+
|dim_categoria_construccion|9                 |
|dim_cielorraso            |2                 |
|dim_cubierta              |5                 |
|dim_departamento          |19                |
|dim_destino               |72                |
|dim_estado_conservacion   |9                 |
|dim_localidad             |498               |
|dim_tipo_obra             |26                |
|fact_historico_valores    |1668657           |
|fact_lineas_construccion  |4478065           |
|fact_padrones_rurales     |248022            |
|fact_padrones_urbanos     |1468837           |
+--------------------------+------------------+



## 8. Consultas de validación sobre Hive

Se ejecutan consultas SQL simples sobre las tablas Hive para verificar que el modelo puede ser consultado correctamente.

Estas consultas no corresponden todavía a las preguntas analíticas finales, sino que sirven como prueba de disponibilidad del modelo.

In [9]:
spark.sql("""
SELECT 
    d.departamento,
    COUNT(*) AS cantidad_padrones_urbanos,
    SUM(p.valor_catastral_total) AS valor_catastral_total
FROM obligatorio_catastro.fact_padrones_urbanos p
JOIN obligatorio_catastro.dim_departamento d
    ON p.codigo_departamento = d.codigo_departamento
GROUP BY d.departamento
ORDER BY cantidad_padrones_urbanos DESC
""").show(20, truncate=False)

[Stage 52:>                                                         (0 + 2) / 2]

+--------------+-------------------------+---------------------+
|departamento  |cantidad_padrones_urbanos|valor_catastral_total|
+--------------+-------------------------+---------------------+
|MONTEVIDEO    |471929                   |1200451164942        |
|CANELONES     |232253                   |258014601023         |
|MALDONADO     |172651                   |372452695003         |
|ROCHA         |100260                   |33418492167          |
|COLONIA       |67085                    |82437017561          |
|SALTO         |44086                    |56291495931          |
|PAYSANDU      |43994                    |46403041948          |
|SAN JOSE      |41709                    |15820146422          |
|CERRO LARGO   |35962                    |28182993734          |
|RIVERA        |35069                    |32882516560          |
|TACUAREMBO    |32557                    |28292643157          |
|FLORIDA       |30107                    |6474477512           |
|LAVALLEJA     |28843    

In [10]:
spark.sql("""
SELECT 
    c.cielorraso,
    COUNT(*) AS cantidad_lineas,
    SUM(l.area_construida) AS area_construida_total
FROM obligatorio_catastro.fact_lineas_construccion l
LEFT JOIN obligatorio_catastro.dim_cielorraso c
    ON l.codigo_cielorraso = c.codigo_cielorraso
GROUP BY c.cielorraso
ORDER BY cantidad_lineas DESC
""").show(truncate=False)

[Stage 56:>                                                         (0 + 2) / 2]

+-----------------------------+---------------+---------------------+
|cielorraso                   |cantidad_lineas|area_construida_total|
+-----------------------------+---------------+---------------------+
|Sin cielorraso / No informado|4326645        |163572859            |
|Con cielorraso               |151420         |8734431              |
+-----------------------------+---------------+---------------------+

